# 06. Pipeline de Transformação Simplificado - Encoding e Scaling

## 🎯 Objetivo

Implementar um **pipeline simplificado de transformação** para preparar os dados
com features de safra (YYYYMMDD) para modelagem.

## 📚 Transformações Aplicadas

### 1. **Seleção de Features**
- Remoção de colunas desnecessárias (IDs, Timestamp original, etc.)
- Manutenção da safra e features temporais

### 2. **Categorical Encoding**
- **One-Hot Encoding**: Variáveis categóricas de baixa cardinalidade
- **Label Encoding**: Variáveis de alta cardinalidade

### 3. **Numerical Transformations**
- **Imputação**: Preencher valores ausentes com mediana
- **Normalização**: StandardScaler para variáveis numéricas

## ⚠️ Garantia Anti-Leakage

✅ **Fit APENAS no treino**: Parâmetros aprendidos somente do treino  
✅ **Transform em treino e OOT**: Usa parâmetros do treino  
✅ **Pipeline simplificado**: Sem features de velocidade complexas  

---

## 1. Setup - Importações e Configuração

In [26]:
# Configurar path para importar módulos do projeto
import sys
from pathlib import Path

notebook_dir = Path.cwd()
if notebook_dir.name == 'notebooks':
    sys.path.insert(0, str(notebook_dir.parent))

# Importar configurações do projeto
from source.config import PROJ_ROOT, get_data_path, get_model_path, ensure_directories

# Garantir que os diretórios existam
ensure_directories()

print(f"✅ Projeto Root: {PROJ_ROOT}")
print(f"✅ Configurações carregadas de source/config.py")

✅ Projeto Root: C:\Users\win\OneDrive\Área de Trabalho\TCC\money_laundering
✅ Configurações carregadas de source/config.py


In [27]:
# Importações padrão
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from datetime import datetime
import json
import joblib

# Scikit-Learn
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# Configurações
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
pd.set_option('display.max_columns', None)

print("✅ Bibliotecas importadas com sucesso!")
print("✅ Pipeline Simplificado!")

✅ Bibliotecas importadas com sucesso!
✅ Pipeline Simplificado!


## 2. Carregamento dos Dados com Features Simplificadas

Carregamos os dados **COM features de safra** gerados no Notebook 05.

In [30]:
print("="*80)
print("CARREGAMENTO DOS DADOS COM FEATURES DE SAFRA")
print("="*80)

import gc

# Colunas pesadas que serão removidas depois (já excluímos na leitura)
DROP_AT_SOURCE = {
    'From Bank', 'To Bank',
    'From Account', 'To Account',
    'From Entity ID', 'To Entity ID'
}

def _build_usecols(csv_path):
    cols = pd.read_csv(csv_path, nrows=0).columns.tolist()
    return [c for c in cols if c not in DROP_AT_SOURCE]

def _estimate_dtypes(csv_path, usecols, sample_rows=200_000):
    sample = pd.read_csv(
        csv_path,
        nrows=sample_rows,
        usecols=usecols,
        low_memory=True
    )

    dtype_map = {}
    for col in sample.columns:
        if col == 'Is Laundering':
            dtype_map[col] = 'int8'
            continue
        if col == 'safra':
            dtype_map[col] = 'int32'
            continue

        if pd.api.types.is_integer_dtype(sample[col]):
            cmin, cmax = sample[col].min(), sample[col].max()
            if cmin >= 0 and cmax <= np.iinfo(np.uint16).max:
                dtype_map[col] = 'uint16'
            elif cmin >= 0 and cmax <= np.iinfo(np.uint32).max:
                dtype_map[col] = 'uint32'
            elif cmin >= np.iinfo(np.int16).min and cmax <= np.iinfo(np.int16).max:
                dtype_map[col] = 'int16'
            elif cmin >= np.iinfo(np.int32).min and cmax <= np.iinfo(np.int32).max:
                dtype_map[col] = 'int32'
        elif pd.api.types.is_float_dtype(sample[col]):
            dtype_map[col] = 'float32'

    del sample
    gc.collect()
    return dtype_map

def _read_csv_memory_safe(csv_path, dataset_name, chunksize=300_000, fallback_sample_frac=0.35):
    usecols = _build_usecols(csv_path)
    dtype_map = _estimate_dtypes(csv_path, usecols)

    print(f"\n📥 Carregando {dataset_name}...")
    print(f"   - Colunas lidas: {len(usecols)}")
    print(f"   - Chunk size: {chunksize:,}")

    chunks = []
    total_rows = 0

    reader = pd.read_csv(
        csv_path,
        usecols=usecols,
        dtype=dtype_map,
        chunksize=chunksize,
        low_memory=True,
        memory_map=True
    )

    for i, chunk in enumerate(reader, start=1):
        if 'Timestamp' in chunk.columns:
            chunk['Timestamp'] = pd.to_datetime(chunk['Timestamp'], errors='coerce', format='mixed')

        # Downcast extra para colunas que não entraram no dtype_map
        f64_cols = chunk.select_dtypes(include=['float64']).columns
        if len(f64_cols) > 0:
            chunk[f64_cols] = chunk[f64_cols].astype('float32')

        i64_cols = chunk.select_dtypes(include=['int64']).columns
        for col in i64_cols:
            chunk[col] = pd.to_numeric(chunk[col], downcast='integer')

        total_rows += len(chunk)
        chunks.append(chunk)

        if i % 10 == 0:
            print(f"   - Chunks processados: {i} | Linhas lidas: {total_rows:,}")

    try:
        df = pd.concat(chunks, ignore_index=True, copy=False)
    except MemoryError:
        print("\n⚠️ MemoryError ao consolidar todos os chunks.")
        print(f"⚠️ Aplicando fallback com amostragem de {fallback_sample_frac*100:.0f}% para continuar o notebook.")
        sampled_chunks = [c.sample(frac=fallback_sample_frac, random_state=42) for c in chunks]
        df = pd.concat(sampled_chunks, ignore_index=True, copy=False)

    del chunks
    gc.collect()

    invalid_ts = df['Timestamp'].isna().sum() if 'Timestamp' in df.columns else 0
    if invalid_ts > 0:
        print(f"   - ⚠️ Timestamps inválidos (NaT): {invalid_ts:,}")

    print(f"   - ✅ {dataset_name} carregado: {df.shape}")
    print(f"   - Memória: {df.memory_usage(deep=True).sum() / (1024**2):.2f} MB")
    return df

# Carregar datasets com estratégia anti-MemoryError
train_path = get_data_path('df_treino_with_features.csv', 'processed')
oot_path = get_data_path('df_oot_with_features.csv', 'processed')

df_treino = _read_csv_memory_safe(train_path, 'Treino')
df_oot = _read_csv_memory_safe(oot_path, 'OOT')

print(f"\n📊 Dataset de Treino:")
print(f"   Shape: {df_treino.shape}")
print(f"   Período: {df_treino['Timestamp'].min()} até {df_treino['Timestamp'].max()}")
print(f"   Taxa de lavagem: {df_treino['Is Laundering'].mean()*100:.2f}%")

print(f"\n📊 Dataset de OOT:")
print(f"   Shape: {df_oot.shape}")
print(f"   Período: {df_oot['Timestamp'].min()} até {df_oot['Timestamp'].max()}")
print(f"   Taxa de lavagem: {df_oot['Is Laundering'].mean()*100:.2f}%")

print(f"\n✅ Dados carregados com sucesso!")

# Mostrar safras
print(f"\n📅 Safras disponíveis:")
print(f"   Treino: {df_treino['safra'].min()} até {df_treino['safra'].max()}")
print(f"   Treino - Safras únicas: {df_treino['safra'].nunique()}")
print(f"   OOT: {df_oot['safra'].min()} até {df_oot['safra'].max()}")
print(f"   OOT - Safras únicas: {df_oot['safra'].nunique()}")

CARREGAMENTO DOS DADOS COM FEATURES DE SAFRA

📥 Carregando Treino...
   - Colunas lidas: 30
   - Chunk size: 100,000
   - Limite de linhas para execução: 3,000,000
   - Chunks processados: 20 | Linhas lidas: 2,000,000 | Linhas mantidas: 2,000,000
   - Chunks processados: 30 | Linhas lidas: 3,000,000 | Linhas mantidas: 3,000,000
   - ⚠️ Dataset reduzido para execução estável: 3,000,000 linhas
   - ✅ Treino carregado: (3000000, 30)
   - Memória: 251.77 MB

📥 Carregando OOT...
   - Colunas lidas: 30
   - Chunk size: 100,000
   - Limite de linhas para execução: 1,500,000
   - Chunks processados: 15 | Linhas lidas: 1,500,000 | Linhas mantidas: 1,500,000
   - ⚠️ Dataset reduzido para execução estável: 1,500,000 linhas
   - ✅ OOT carregado: (1500000, 30)
   - Memória: 125.89 MB

📊 Dataset de Treino:
   Shape: (3000000, 30)
   Período: 2022-09-01 00:00:00 até 2022-09-01 13:33:00
   Taxa de lavagem: 0.02%

📊 Dataset de OOT:
   Shape: (1500000, 30)
   Período: 2022-09-14 05:39:00 até 2022-09-15 

## 3. Separação de Features e Target

Separamos X (features) e y (target) removendo colunas desnecessárias.

In [31]:
# Definir target
target_col = 'Is Laundering'

# Colunas para remover (além do target)
cols_to_remove = [
    target_col,
    'Timestamp',  # Já temos safra e componentes temporais
    'From Bank', 'To Bank',  # Alta cardinalidade, usar apenas se necessário
    'From Account', 'To Account',  # IDs individuais
    'From Entity ID', 'To Entity ID',  # IDs de entidade
]

# Verificar quais colunas existem antes de remover
cols_to_remove = [col for col in cols_to_remove if col in df_treino.columns]

# Separar X e y
X_train = df_treino.drop(columns=cols_to_remove)
y_train = df_treino[target_col]

X_oot = df_oot.drop(columns=cols_to_remove)
y_oot = df_oot[target_col]

print("="*80)
print("SEPARAÇÃO DE FEATURES E TARGET")
print("="*80)

print(f"\n📊 Colunas removidas ({len(cols_to_remove)}):")
for col in cols_to_remove:
    print(f"   - {col}")

print(f"\n📊 Treino:")
print(f"   X_train shape: {X_train.shape}")
print(f"   y_train shape: {y_train.shape}")
print(f"   Taxa de lavagem: {y_train.mean()*100:.2f}%")

print(f"\n📊 OOT:")
print(f"   X_oot shape: {X_oot.shape}")
print(f"   y_oot shape: {y_oot.shape}")
print(f"   Taxa de lavagem: {y_oot.mean()*100:.2f}%")

print(f"\n📋 Features disponíveis ({X_train.shape[1]}):")
print(f"   Principais colunas:")
for col in X_train.columns[:15]:
    print(f"   - {col}")
if X_train.shape[1] > 15:
    print(f"   ... e mais {X_train.shape[1] - 15} colunas")

SEPARAÇÃO DE FEATURES E TARGET

📊 Colunas removidas (2):
   - Is Laundering
   - Timestamp

📊 Treino:
   X_train shape: (3000000, 28)
   y_train shape: (3000000,)
   Taxa de lavagem: 0.02%

📊 OOT:
   X_oot shape: (1500000, 28)
   y_oot shape: (1500000,)
   Taxa de lavagem: 0.05%

📋 Features disponíveis (28):
   Principais colunas:
   - Amount Received
   - Receiving Currency
   - Amount Paid
   - Payment Currency
   - Payment Format
   - Bank ID
   - Bank ID_To
   - safra
   - year
   - month
   - day
   - hour
   - dayofweek
   - quarter
   - month_sin
   ... e mais 13 colunas


## 4. Identificação de Colunas para Pipeline

Identificar colunas categóricas e numéricas para aplicar transformações adequadas.

In [32]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

# Identificar colunas categóricas e numéricas
categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
# Inclui int/float em qualquer precisão (int8/int16/int32/int64/float32/float64)
numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()

# Remover target se aparecer por segurança
if 'Is Laundering' in numeric_cols:
    numeric_cols.remove('Is Laundering')

# Remover safra se estiver nos numéricos (é identificador, não feature)
if 'safra' in numeric_cols:
    numeric_cols.remove('safra')

print("\n" + "="*80)
print("IDENTIFICAÇÃO DE COLUNAS")
print("="*80)

print(f"\n📊 Colunas categóricas ({len(categorical_cols)}):")
for col in categorical_cols[:10]:
    nunique = X_train[col].nunique()
    print(f"   - {col}: {nunique} categorias")
if len(categorical_cols) > 10:
    print(f"   ... e mais {len(categorical_cols) - 10} colunas")

# Separar categóricas por cardinalidade
low_cardinality = []
high_cardinality = []
for col in categorical_cols:
    nunique = X_train[col].nunique()
    if nunique <= 20:
        low_cardinality.append(col)
    else:
        high_cardinality.append(col)

print(f"\n📊 Categóricas de baixa cardinalidade (≤20): {len(low_cardinality)}")
for col in low_cardinality:
    print(f"   - {col}: {X_train[col].nunique()} categorias")

print(f"\n📊 Categóricas de alta cardinalidade (>20): {len(high_cardinality)}")
for col in high_cardinality:
    print(f"   - {col}: {X_train[col].nunique()} categorias -> REMOVIDA")

# Ajustar categóricas para usar apenas baixa cardinalidade
categorical_cols = low_cardinality

print(f"\n📊 Colunas numéricas ({len(numeric_cols)}):")
for col in numeric_cols[:10]:
    print(f"   - {col}")
if len(numeric_cols) > 10:
    print(f"   ... e mais {len(numeric_cols) - 10} colunas")


IDENTIFICAÇÃO DE COLUNAS

📊 Colunas categóricas (3):
   - Receiving Currency: 15 categorias
   - Payment Currency: 15 categorias
   - Payment Format: 7 categorias

📊 Categóricas de baixa cardinalidade (≤20): 3
   - Receiving Currency: 15 categorias
   - Payment Currency: 15 categorias
   - Payment Format: 7 categorias

📊 Categóricas de alta cardinalidade (>20): 0

📊 Colunas numéricas (24):
   - Amount Received
   - Amount Paid
   - Bank ID
   - Bank ID_To
   - year
   - month
   - day
   - hour
   - dayofweek
   - quarter
   ... e mais 14 colunas


## 5. Construção do Pipeline Simplificado

Criar pipeline com transformadores para colunas numéricas e categóricas.

In [33]:
# Pipeline para colunas numéricas
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Pipeline para colunas categóricas
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=True))
])

# Combinar transformadores
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols)
    ],
    remainder='drop'
 )

print("\n" + "="*80)
print("PIPELINE CONSTRUÍDO")
print("="*80)

print(f"\n📦 Transformadores:")
print(f"   - Numérico: SimpleImputer(median) + StandardScaler")
print(f"   - Categórico: SimpleImputer(missing) + OneHotEncoder (sparse)")

print(f"\n✓ Pipeline pronto para fit/transform")


PIPELINE CONSTRUÍDO

📦 Transformadores:
   - Numérico: SimpleImputer(median) + StandardScaler
   - Categórico: SimpleImputer(missing) + OneHotEncoder (sparse)

✓ Pipeline pronto para fit/transform


## 6. Aplicação do Pipeline - Fit e Transform

⚠️ **CRÍTICO**: Fit APENAS no X_train, Transform em X_train e X_oot.

In [34]:
# Fit no treino e transform em ambos (com proteção contra MemoryError)
from scipy import sparse

print("\n" + "="*80)
print("FIT E TRANSFORM")
print("="*80)

def _safe_feature_names(preprocessor_obj, numeric_cols_local, categorical_cols_local, n_output_cols):
    try:
        names = preprocessor_obj.get_feature_names_out().tolist()
        names = [n.replace('num__', '').replace('cat__', '') for n in names]
    except Exception:
        names = list(numeric_cols_local)
        if len(categorical_cols_local) > 0:
            try:
                cat_encoder = preprocessor_obj.named_transformers_['cat'].named_steps['onehot']
                cat_names = cat_encoder.get_feature_names_out(categorical_cols_local).tolist()
                names.extend(cat_names)
            except Exception:
                pass

    # Garantir compatibilidade entre número de nomes e colunas de saída
    if len(names) != n_output_cols:
        print(
            f"⚠️ Ajustando nomes de features: nomes={len(names)} | colunas_transformadas={n_output_cols}"
        )
        if len(names) > n_output_cols:
            names = names[:n_output_cols]
        else:
            names.extend([f"feature_{i}" for i in range(len(names), n_output_cols)])

    return names

def _to_dataframe_safe(X, feature_names, index, name):
    if sparse.issparse(X):
        density = X.nnz / (X.shape[0] * X.shape[1]) if X.shape[0] and X.shape[1] else 0.0
        print(f"   - {name} em formato esparso | densidade: {density:.6f}")
        return pd.DataFrame.sparse.from_spmatrix(X, index=index, columns=feature_names)

    if hasattr(X, 'dtype') and X.dtype == np.float64:
        X = X.astype(np.float32)
    return pd.DataFrame(X, columns=feature_names, index=index)

try:
    print("\n⏳ Fitting no treino...")
    preprocessor.fit(X_train)
    print("✓ Fit concluído!")

    print("\n⏳ Transforming treino...")
    X_train_transformed_raw = preprocessor.transform(X_train)

    print("\n⏳ Transforming OOT...")
    X_oot_transformed_raw = preprocessor.transform(X_oot)

except MemoryError:
    print("\n⚠️ MemoryError durante fit/transform com configuração atual.")
    print("⚠️ Reconfigurando OneHotEncoder para saída esparsa e tentando novamente...")

    if len(categorical_cols) > 0:
        preprocessor.set_params(cat__onehot__sparse_output=True)

    print("\n⏳ Re-fit no treino (modo esparso)...")
    preprocessor.fit(X_train)
    print("✓ Re-fit concluído!")

    print("\n⏳ Re-transform treino...")
    X_train_transformed_raw = preprocessor.transform(X_train)

    print("\n⏳ Re-transform OOT...")
    X_oot_transformed_raw = preprocessor.transform(X_oot)

print("\n✓ Transform concluído!")

n_output_cols = X_train_transformed_raw.shape[1]
feature_names = _safe_feature_names(
    preprocessor, numeric_cols, categorical_cols, n_output_cols
)

X_train_transformed = _to_dataframe_safe(
    X_train_transformed_raw, feature_names, X_train.index, 'Treino'
 )
X_oot_transformed = _to_dataframe_safe(
    X_oot_transformed_raw, feature_names, X_oot.index, 'OOT'
 )

print("\n" + "="*80)
print("TRANSFORMAÇÃO CONCLUÍDA")
print("="*80)

print(f"\n📊 Shapes transformados:")
print(f"   X_train: {X_train.shape} → {X_train_transformed.shape}")
print(f"   X_oot: {X_oot.shape} → {X_oot_transformed.shape}")

print(f"\n📝 Features finais: {X_train_transformed.shape[1]}")


FIT E TRANSFORM

⏳ Fitting no treino...
✓ Fit concluído!

⏳ Transforming treino...

⏳ Transforming OOT...

✓ Transform concluído!

TRANSFORMAÇÃO CONCLUÍDA

📊 Shapes transformados:
   X_train: (3000000, 28) → (3000000, 61)
   X_oot: (1500000, 28) → (1500000, 61)

📝 Features finais: 61


In [35]:
# Estatísticas descritivas (compatível com DataFrame esparso e otimizado por amostragem)
print("="*80)
print("ESTATÍSTICAS DESCRITIVAS - DADOS TRANSFORMADOS (Treino)")
print("="*80)

def _to_dense_numeric(df, cols):
    sub = df[cols].copy()
    for c in sub.columns:
        if pd.api.types.is_sparse(sub[c].dtype):
            sub[c] = sub[c].sparse.to_dense()
    return sub.apply(pd.to_numeric, errors='coerce').astype('float32')

sample_n = min(200_000, len(X_train_transformed))
if sample_n < len(X_train_transformed):
    stats_base = X_train_transformed.sample(n=sample_n, random_state=42)
    print(f"\n📌 Estatísticas calculadas em amostra de {sample_n:,} linhas.")
else:
    stats_base = X_train_transformed

print(f"\n📊 Shape total: {X_train_transformed.shape}")
print(f"\n📊 Primeiras features:")
first_cols = stats_base.columns[:10].tolist()
first_dense = _to_dense_numeric(stats_base, first_cols)
print(first_dense.describe().T)

print(f"\n📊 Últimas features:")
if stats_base.shape[1] > 10:
    last_cols = stats_base.columns[-10:].tolist()
    last_dense = _to_dense_numeric(stats_base, last_cols)
    print(last_dense.describe().T)

ESTATÍSTICAS DESCRITIVAS - DADOS TRANSFORMADOS (Treino)

📌 Estatísticas calculadas em amostra de 200,000 linhas.

📊 Shape total: (3000000, 61)

📊 Primeiras features:
                    count      mean       std       min       25%       50%  \
Amount Received  200000.0  0.004173  1.671042 -0.004716 -0.004716 -0.004715   
Amount Paid      200000.0  0.005077  2.046744 -0.004381 -0.004381 -0.004380   
Bank ID          200000.0 -0.004434  0.995059 -0.533415 -0.512021 -0.360301   
Bank ID_To       200000.0  0.000191  1.000221 -0.607892 -0.563392 -0.396274   
year             200000.0  0.000000  0.000000  0.000000  0.000000  0.000000   
month            200000.0  0.000000  0.000000  0.000000  0.000000  0.000000   
day              200000.0  0.000000  0.000000  0.000000  0.000000  0.000000   
hour             200000.0 -0.001397  0.999845 -0.863505 -0.863505 -0.399908   
dayofweek        200000.0  0.000000  0.000000  0.000000  0.000000  0.000000   
quarter          200000.0  0.000000  0.00000

In [36]:
# Verificar valores ausentes
print("="*80)
print("VALORES AUSENTES APÓS TRANSFORMAÇÃO")
print("="*80)

nan_train = X_train_transformed.isnull().sum().sum()
nan_oot = X_oot_transformed.isnull().sum().sum()

print(f"\n✓ Treino: {nan_train} valores ausentes")
print(f"✓ OOT: {nan_oot} valores ausentes")

if nan_train == 0 and nan_oot == 0:
    print("\n✅ Nenhum valor ausente! Pipeline de imputação funcionou corretamente.")
else:
    print("\n⚠️  Ainda há valores ausentes. Verificar pipeline de imputação.")

VALORES AUSENTES APÓS TRANSFORMAÇÃO

✓ Treino: 0 valores ausentes
✓ OOT: 0 valores ausentes

✅ Nenhum valor ausente! Pipeline de imputação funcionou corretamente.


## 8. Validação da Normalização

Verificar se as features numéricas estão normalizadas (média ~0, std ~1).

In [37]:
print("="*80)
print("VALIDAÇÃO DA NORMALIZAÇÃO (StandardScaler)")
print("="*80)

numeric_cols_present = [c for c in numeric_cols if c in X_train_transformed.columns]

if len(numeric_cols_present) == 0:
    print("\n⚠️ Nenhuma coluna numérica original encontrada para validação.")
else:
    sample_n = min(300_000, len(X_train_transformed))
    if sample_n < len(X_train_transformed):
        numeric_features = X_train_transformed[numeric_cols_present].sample(n=sample_n, random_state=42).copy()
        print(f"\n📌 Validação calculada em amostra de {sample_n:,} linhas.")
    else:
        numeric_features = X_train_transformed[numeric_cols_present].copy()

    for c in numeric_features.columns:
        if pd.api.types.is_sparse(numeric_features[c].dtype):
            numeric_features[c] = numeric_features[c].sparse.to_dense()

    numeric_features = numeric_features.apply(pd.to_numeric, errors='coerce').astype('float32')

    means = numeric_features.mean()
    stds = numeric_features.std(ddof=0)

    print(f"\n📊 Médias (devem estar próximas de 0):")
    print(f"   Min: {means.min():.4f}")
    print(f"   Max: {means.max():.4f}")
    print(f"   Média das médias: {means.mean():.4f}")

    print(f"\n📊 Desvios padrão (devem estar próximos de 1):")
    print(f"   Min: {stds.min():.4f}")
    print(f"   Max: {stds.max():.4f}")
    print(f"   Média dos stds: {stds.mean():.4f}")

    if abs(means.mean()) < 0.1 and abs(stds.mean() - 1.0) < 0.2:
        print("\n✅ Normalização bem-sucedida!")
    else:
        print("\n⚠️ Normalização pode ter problemas. Revisar pipeline.")

VALIDAÇÃO DA NORMALIZAÇÃO (StandardScaler)

📌 Validação calculada em amostra de 300,000 linhas.

📊 Médias (devem estar próximas de 0):
   Min: -0.0035
   Max: 0.0027
   Média das médias: 0.0001

📊 Desvios padrão (devem estar próximos de 1):
   Min: 0.0000
   Max: 1.6749
   Média dos stds: 0.5018

⚠️ Normalização pode ter problemas. Revisar pipeline.


## 9. Validação Anti-Leakage

Verificar que não há vazamento de informação do OOT para o treino.

In [38]:
print("="*80)
print("VALIDAÇÃO ANTI-LEAKAGE")
print("="*80)

sample_cols = [c for c in numeric_cols[:10] if c in X_train_transformed.columns and c in X_oot_transformed.columns]

if len(sample_cols) == 0:
    print("\n⚠️ Sem colunas numéricas em comum para validar anti-leakage.")
else:
    sample_n_train = min(300_000, len(X_train_transformed))
    sample_n_oot = min(300_000, len(X_oot_transformed))

    if sample_n_train < len(X_train_transformed):
        train_sub = X_train_transformed[sample_cols].sample(n=sample_n_train, random_state=42).copy()
    else:
        train_sub = X_train_transformed[sample_cols].copy()

    if sample_n_oot < len(X_oot_transformed):
        oot_sub = X_oot_transformed[sample_cols].sample(n=sample_n_oot, random_state=42).copy()
    else:
        oot_sub = X_oot_transformed[sample_cols].copy()

    print(f"\n📌 Validação em amostras: treino={len(train_sub):,} | oot={len(oot_sub):,}")

    for c in sample_cols:
        if pd.api.types.is_sparse(train_sub[c].dtype):
            train_sub[c] = train_sub[c].sparse.to_dense()
        if pd.api.types.is_sparse(oot_sub[c].dtype):
            oot_sub[c] = oot_sub[c].sparse.to_dense()

    train_sub = train_sub.apply(pd.to_numeric, errors='coerce').astype('float32')
    oot_sub = oot_sub.apply(pd.to_numeric, errors='coerce').astype('float32')

    train_means = train_sub.mean()
    oot_means = oot_sub.mean()

    train_stds = train_sub.std(ddof=0)
    oot_stds = oot_sub.std(ddof=0)

    diff_means = abs(train_means - oot_means).mean()
    diff_stds = abs(train_stds - oot_stds).mean()

    print("\n✓ Verificação: Estatísticas do OOT diferem do treino?")
    print(f"\n  Diferença média das médias: {diff_means:.4f}")
    print(f"  Diferença média dos stds: {diff_stds:.4f}")

    if diff_means > 0.01:
        print("\n  ✅ OOT tem estatísticas diferentes -> Pipeline aplicado corretamente!")
    else:
        print("\n  ⚠️ OOT tem estatísticas muito similares -> Possível leakage!")

print("\n✅ Validação anti-leakage concluída!")

VALIDAÇÃO ANTI-LEAKAGE

📌 Validação em amostras: treino=300,000 | oot=300,000

✓ Verificação: Estatísticas do OOT diferem do treino?

  Diferença média das médias: 1.6506
  Diferença média dos stds: 0.2887

  ✅ OOT tem estatísticas diferentes -> Pipeline aplicado corretamente!

✅ Validação anti-leakage concluída!


## 10. Salvamento dos Dados Transformados

Salvamos X_train, X_oot, y_train, y_oot e o pipeline para uso no treinamento de modelos.

In [39]:
print("="*80)
print("SALVAMENTO DOS DADOS TRANSFORMADOS")
print("="*80)

# Definir caminhos
path_X_train = get_data_path('X_train.csv', 'processed')
path_X_oot = get_data_path('X_oot.csv', 'processed')
path_y_train = get_data_path('y_train.csv', 'processed')
path_y_oot = get_data_path('y_oot.csv', 'processed')
path_pipeline = get_model_path('preprocessing_pipeline.pkl')

# Salvar dados
X_train_transformed.to_csv(path_X_train, index=False)
X_oot_transformed.to_csv(path_X_oot, index=False)
y_train.to_frame().to_csv(path_y_train, index=False)
y_oot.to_frame().to_csv(path_y_oot, index=False)

# Salvar pipeline
joblib.dump(preprocessor, path_pipeline)

print(f"\n✅ Datasets e pipeline salvos com sucesso!")
print(f"\n📁 Arquivos de dados:")
print(f"   - {path_X_train}")
print(f"   - {path_X_oot}")
print(f"   - {path_y_train}")
print(f"   - {path_y_oot}")

print(f"\n📁 Pipeline:")
print(f"   - {path_pipeline}")

# Verificar tamanhos
size_X_train = Path(path_X_train).stat().st_size / (1024 * 1024)
size_X_oot = Path(path_X_oot).stat().st_size / (1024 * 1024)

print(f"\n📏 Tamanhos:")
print(f"   - X_train: {size_X_train:.2f} MB")
print(f"   - X_oot: {size_X_oot:.2f} MB")

SALVAMENTO DOS DADOS TRANSFORMADOS

✅ Datasets e pipeline salvos com sucesso!

📁 Arquivos de dados:
   - C:\Users\win\OneDrive\Área de Trabalho\TCC\money_laundering\data\processed\X_train.csv
   - C:\Users\win\OneDrive\Área de Trabalho\TCC\money_laundering\data\processed\X_oot.csv
   - C:\Users\win\OneDrive\Área de Trabalho\TCC\money_laundering\data\processed\y_train.csv
   - C:\Users\win\OneDrive\Área de Trabalho\TCC\money_laundering\data\processed\y_oot.csv

📁 Pipeline:
   - C:\Users\win\OneDrive\Área de Trabalho\TCC\money_laundering\models\preprocessing_pipeline.pkl

📏 Tamanhos:
   - X_train: 935.85 MB
   - X_oot: 504.79 MB


## 11. Teste de Carregamento do Pipeline

Verificar que o pipeline salvo pode ser recarregado e usado corretamente.

In [40]:
print("="*80)
print("TESTE DE CARREGAMENTO DO PIPELINE")
print("="*80)

# Carregar pipeline salvo
pipeline_loaded = joblib.load(get_model_path('preprocessing_pipeline.pkl'))

print(f"\n✅ Pipeline carregado com sucesso!")

# Testar transform em amostra
sample = X_train.head(10)
transformed_sample = pipeline_loaded.transform(sample)

print(f"\n✓ Teste de transform:")
print(f"   Input shape: {sample.shape}")
print(f"   Output shape: {transformed_sample.shape}")

print("\n✅ Pipeline salvo está funcional!")

TESTE DE CARREGAMENTO DO PIPELINE

✅ Pipeline carregado com sucesso!

✓ Teste de transform:
   Input shape: (10, 28)
   Output shape: (10, 61)

✅ Pipeline salvo está funcional!


## 12. Sumário e Próximos Passos

### ✅ Realizações deste Notebook

1. ✅ Construção de **Pipeline sklearn** simplificado
2. ✅ Aplicação de **Fit APENAS no treino**
3. ✅ Transformação consistente em **treino e OOT**
4. ✅ **Categorical encoding** (One-Hot)
5. ✅ **Imputação** (mediana para numéricos, constante para categóricos)
6. ✅ **Normalização** (StandardScaler ajustado no treino)
7. ✅ Validação **anti-leakage**
8. ✅ Persistência do pipeline e datasets

### 📊 Estatísticas Finais

- **Samples treino**: {X_train_transformed.shape[0]:,}
- **Samples OOT**: {X_oot_transformed.shape[0]:,}
- **Features finais**: {X_train_transformed.shape[1]}

### 🔄 Próximos Passos

**Notebook 08**: Treinamento de Modelos com:
- TimeSeriesSplit para validação temporal
- Balanceamento correto (dentro dos folds)
- Threshold otimizado

---

**Data**: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}  
**Status**: ✅ Pipeline Simplificado Concluído